# Module 3 — Robust Evaluation

**By the end of this notebook, you will be able to:**
- Explain why a single manual train/test pair gives a fragile, direction-dependent performance estimate
- Use `GroupKFold` to evaluate a model against every group in turn, without ever mixing a group across train and validation
- Read a mean **and** a std across folds, not just a single number

**Context:** In Session 2's reflection, you compared training on Kampala and testing on Nairobi, then reversed it — and got two different, both disappointing results. That comparison only used two of the four cities, and only one direction each. This module systematizes it: evaluate against every city, one at a time, on the full 4-city dataset you cleaned in Module 2.

## Rebuild the cleaned dataset

**Exercise:** Reuse the functions you completed in Module 2 to rebuild `enriched`: scope `train_df` to all four cities and `DEFAULT_COLUMNS`, drop columns above a 0.7 missing-value threshold, fill what remains per city, then add temporal features — the exact same sequence as `workflows.run_advanced`, minus the final split.

In [ ]:
# TODO: rebuild enriched, reusing the Module 2 functions you already completed
from sklearn.linear_model import LinearRegression
import pandas as pd

from air_quality import data, evaluation, features
from air_quality.workflows import DEFAULT_COLUMNS

ALL_CITIES = ["Kampala", "Nairobi", "Lagos", "Bujumbura"]
train_df, _ = data.load_datasets()
train_df["city"].value_counts()

city
Kampala      5596
Nairobi      1500
Lagos         852
Bujumbura     123
Name: count, dtype: int64

In [8]:
scoped = data.restrict_to_scope(train_df, ALL_CITIES, DEFAULT_COLUMNS)

sparse_cols = data.columns_above_missing_threshold(scoped, threshold=0.7)
cleaned = data.drop_columns(scoped, sparse_cols)

cols_to_fill = cleaned.columns[cleaned.isna().any()].tolist()
filled = data.fill_missing_by_city(cleaned, cols_to_fill)

enriched = features.add_temporal_features(filled)

feature_cols = [c for c in enriched.columns if c not in ["city", "date", "pm2_5"]]

## One pair only tells you about that pair

Recall Session 2's numbers:

| Direction | RMSE | R² |
|---|---|---|
| Train Kampala → test Nairobi | 24.6 | -0.02 |
| Train Nairobi → test Kampala | 15.2 | -0.16 |

Neither number tells you anything about Lagos or Bujumbura — they never appeared in either run. Would the model do better or worse there? You cannot know from a single manual pair, and picking a *different* pair would have given you yet another, equally partial answer.

[`GroupKFold`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GroupKFold.html) automates the *idea* behind the manual split — never mix a city across train and validation — but applies it systematically: with one fold per city, each city takes a turn being the one the model has never seen, while training on the other three. This is sometimes called leave-one-group-out.

In [9]:
# TODO 3: drop columns missing more than 70% of the time, then fill the
# rest per city, and confirm nothing is missing anymore
sparse_cols = data.columns_above_missing_threshold(scoped, threshold=0.7)
cleaned = data.drop_columns(scoped, sparse_cols)

cols_to_fill = cleaned.columns[cleaned.isna().any()].tolist()
filled = data.fill_missing_by_city(cleaned, cols_to_fill)

filled.isna().sum().sum() 

np.int64(0)

## Implement it

Open `src/air_quality/evaluation.py` and complete `evaluate_group_cv`, in the "Session 3 — optional modules" section at the bottom of the file. Its docstring specifies the exact contract, and `tests/test_evaluation_advanced.py` specifies the exact expected behavior — read it the same way you read `tests/test_evaluation.py` in Session 2. One detail the docstring calls out: fit a fresh [`clone()`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html) of the model for each fold, not the same instance refit repeatedly — a fold must never carry over anything learned by a previous fold. Run `uv run pytest tests/test_evaluation_advanced.py -v` until it passes, then come back here.

## Apply it, on all four cities at once

**Exercise:** Call `evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")`, then look at the per-fold breakdown as a table.

In [10]:
# Given: once evaluate_group_cv is implemented, this just calls it
result = evaluation.evaluate_group_cv(LinearRegression(), enriched, feature_cols, groups_col="city")

pd.DataFrame(result["folds"]).set_index("test_group")

,rmse,mae,r2
test_group,,,
Kampala,13.199477,9.224265,0.153398
Nairobi,25.281721,10.135720,-0.024804
Lagos,53.070013,22.524251,-0.009762
Bujumbura,23.733924,20.937887,-0.136574


In [11]:
# Given: the aggregated view
{k: v for k, v in result.items() if k != "folds"}

{'rmse_mean': 28.82128376331335,
 'rmse_std': 14.751707260827622,
 'mae_mean': 15.70553063810373,
 'mae_std': 6.060158811745567,
 'r2_mean': -0.004435557025669917,
 'r2_std': 0.10345936519032767}

**Look at the per-city RMSE.** One city is far harder to predict for than the other three — the kind of gap a single Kampala/Nairobi comparison could never have revealed, in either direction.

**Continue on the site:** [Module 3 — Robust Evaluation](https://hub.imt-atlantique.fr/datascience-toolkit/session3/practical_3/)